In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
!pip install dash

FILE = "/content/[5442] EcoMoveMobility.xlsx"

# 1. Loading and Preparing Data
df = pd.read_excel(FILE)

# Converting to real dates
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")

# New columns
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Profit_EUR"] = df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]

print("Rows and columns:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df.head())

# 1b. Calculation and Consistency Checks
for c in ["City", "Country", "Transport_Mode", "Route_Type", "Weather", "Event_Day"]:
    spaces = (df[c] != df[c].str.strip()).sum()                 # leading/trailing spaces
    case_clash = df[c].nunique() - df[c].str.lower().nunique()  # e.g. "bus" and "Bus"
    print(f"{c}: {df[c].nunique()} unique values, {spaces} with extra spaces, {case_clash} case clashes")
print("Each city maps to one country:", df.groupby("City")["Country"].nunique().max() == 1)
original_numbers = [c for c in df.select_dtypes("number").columns if c not in ("Profit_EUR", "Year")]
print("Negative numbers:", int((df[original_numbers] < 0).sum().sum()),
      "| rating range:", df["Customer_Rating"].min(), "-", df["Customer_Rating"].max(),
      "| on-time range:", df["On_Time_Percentage"].min(), "-", df["On_Time_Percentage"].max())

# Delay x Trips
df["Delay_x_Trips"] = df["Average_Delay_Minutes"] * df["Trips"]
df["OnTime_x_Trips"] = df["On_Time_Percentage"] * df["Trips"]
df["Rating_x_Pax"] = df["Customer_Rating"] * df["Passengers"]

def kpis_by(key):
    """Correct group KPIs. Complaints use a ratio of totals, not an average of record rates."""
    g = df.groupby(key)[["Passengers", "Trips", "Delay_x_Trips", "OnTime_x_Trips",
                         "Rating_x_Pax", "Accessibility_Complaints"]].sum()
    return pd.DataFrame({"Avg_Delay_Min": g["Delay_x_Trips"] / g["Trips"],
                         "On_Time_%": g["OnTime_x_Trips"] / g["Trips"],
                         "Rating": g["Rating_x_Pax"] / g["Passengers"],
                         "Complaints_per_1000": g["Accessibility_Complaints"] / g["Passengers"] * 1000}).round(2)

print("\nNetwork delay - simple mean:", round(df["Average_Delay_Minutes"].mean(), 2),
      "| weighted:", kpis_by(lambda i: "Network")["Avg_Delay_Min"].iloc[0])
print("Network on-time % - simple mean:", round(df["On_Time_Percentage"].mean(), 2),
      "| weighted:", kpis_by(lambda i: "Network")["On_Time_%"].iloc[0])
print("\nCorrected KPIs by transport mode:\n", kpis_by("Transport_Mode"))
print("\nCorrected KPIs by weather:\n", kpis_by("Weather"))

# 2. Challenges and Limitations
print("\nRecords per mode:\n", df["Transport_Mode"].value_counts(), "\n")
print("Records per weather type:\n", df["Weather"].value_counts(), "\n")
print("Event-day records:", (df["Event_Day"] == "Yes").sum(), "of", len(df))
print("Unique dates:", df["Date"].nunique(), "of 731 possible days")

q1, q3 = df["Passengers"].quantile([.25, .75])
n = ((df["Passengers"] < q1 - 1.5 * (q3 - q1)) | (df["Passengers"] > q3 + 1.5 * (q3 - q1))).sum()
print(f"Passenger outliers: {n} ({n / len(df):.1%})")

print("Zero-energy records by mode:\n",
      (df["Energy_Consumption_KWh"] == 0).groupby(df["Transport_Mode"]).sum(), "\n")
print("Share of zero-complaint records by mode:\n",
      (df["Accessibility_Complaints"] == 0).groupby(df["Transport_Mode"]).mean().round(2), "\n")

# City ranking robust to uneven record counts?
check = df.groupby("City")["Passengers"].agg(Total="sum", Average_per_record="mean", Records="size").round(0)
check["Rank_total"] = check["Total"].rank(ascending=False).astype(int)
check["Rank_average"] = check["Average_per_record"].rank(ascending=False).astype(int)
print(check.sort_values("Rank_total"))

# 3. Time Series monthly passengers
monthly = df.groupby("Month")["Passengers"].sum().reset_index()
fig1 = px.line(monthly, x="Month", y="Passengers", markers=True,
               title="Monthly Passengers (2024-2025)")
fig1.update_layout(xaxis_title="Month", yaxis_title="Total Passengers")
fig1.show()

# 4. Comparison of passengers by city/country
city = df.groupby(["City", "Country"])["Passengers"].sum().reset_index().sort_values("Passengers")
fig2 = px.bar(city, x="Passengers", y="City", color="Country", orientation="h",
              title="Total Passengers by City/Country", text="Passengers")
fig2.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig2.update_layout(xaxis_title="Total Passengers", yaxis_title="City")
fig2.show()

# 5. Relationsships of correlation heatmap
cols = ["Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage",
        "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
        "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]
corr = df[cols].corr().round(2)
fig3 = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
                 zmin=-1, zmax=1, title="Correlation Heatmap of Key Measures")
fig3.update_layout(width=800, height=700)
fig3.show()

# 6. Distribution delay by transport mode
fig4 = px.box(df, x="Transport_Mode", y="Average_Delay_Minutes", color="Transport_Mode",
              points="outliers", title="Delay Distribution by Transport Mode")
fig4.update_layout(xaxis_title="Transport Mode", yaxis_title="Average Delay (minutes)", showlegend=False)
fig4.show()
print(df.groupby("Transport_Mode")["Average_Delay_Minutes"]
        .describe().round(2).sort_values("50%", ascending=False).to_string())

# 7. Performance Indicator by revenue vs cost (dumbbell)
mode = df.groupby("Transport_Mode")[["Ticket_Revenue_EUR", "Operating_Cost_EUR"]].sum()
mode["Profit_EUR"] = mode["Ticket_Revenue_EUR"] - mode["Operating_Cost_EUR"]
mode["Margin_%"] = (mode["Profit_EUR"] / mode["Ticket_Revenue_EUR"] * 100).round(1)
mode = mode.sort_values("Margin_%")

fig5 = go.Figure()
for m, row in mode.iterrows():  # grey line joins cost to revenue
    fig5.add_trace(go.Scatter(x=[row["Operating_Cost_EUR"], row["Ticket_Revenue_EUR"]], y=[m, m],
                              mode="lines", line=dict(color="grey", width=3), showlegend=False))
fig5.add_trace(go.Scatter(x=mode["Operating_Cost_EUR"], y=mode.index, mode="markers",
                          name="Operating Cost", marker=dict(color="blue", size=14)))
fig5.add_trace(go.Scatter(x=mode["Ticket_Revenue_EUR"], y=mode.index, mode="markers",
                          name="Ticket Revenue", marker=dict(color="orange", size=14)))
fig5.update_layout(title="Revenue vs Operating Cost by Transport Mode (EUR, log scale)",
                   xaxis_title="EUR (log scale)", yaxis_title="Transport Mode", xaxis_type="log")
fig5.show()
print(mode.round(0).to_string())

df["Delay_x_Trips"] = df["Average_Delay_Minutes"] * df["Trips"]
g = df.groupby(["Weather", "Event_Day"])[["Delay_x_Trips", "Trips"]].sum()
w = (g["Delay_x_Trips"] / g["Trips"]).round(2).reset_index(name="Avg_Delay_Min")
fig6 = px.bar(w, x="Weather", y="Avg_Delay_Min", color="Event_Day", barmode="group",
              color_discrete_sequence=["#1f77b4", "#ff7f0e"])
fig6.update_layout(yaxis_title="Average delay per trip (min)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 21.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]
Rows and columns: (5000, 21)
Date range: 2024-01-01 to 2025-12-31
Missing values: 0
Duplicate rows: 0
  Record_ID       Date       City      Country Transport_Mode  \
0  EM-00001 2025-03-31      Paris       France           Tram   
1  EM-00002 2025-03-01      Paris       France           Tram   
2  EM-00003 2025-08-13  Rotterdam  Netherlands           Tram   
3  EM-00004 2024-07-31       Lyon       France           Tram   
4  EM-00005 2024-10-13  Amsterdam  Netherlands          Metro   

          Route_Type Weather Event_Day  Passengers  Trips  ...  \
0       Tourist Zone  Cloudy       Yes       22946    101  ...   
1    University Area  Cloudy        No        9417     59  ...   
2  Airport Connector   Clear        No        6467     38  ...   
3    University Area   Clear       Yes        5218     33  ...   
4       Tourist Zone  Cloudy        No       102

                 count  mean   std  min  25%  50%  75%   max
Transport_Mode                                              
Bus             1375.0  6.58  1.58  1.2  5.5  6.5  7.5  12.3
Park and Ride    585.0  5.17  1.64  0.6  4.0  5.0  6.0  11.5
Tram             761.0  4.79  1.68  0.3  3.6  4.7  5.7  12.5
Metro           1047.0  3.73  1.60  0.0  2.7  3.6  4.6  10.1
E-Scooter        473.0  1.94  1.52  0.0  0.7  1.7  2.9   6.9
Bike Share       759.0  1.62  1.33  0.0  0.5  1.5  2.4   7.3


                Ticket_Revenue_EUR  Operating_Cost_EUR  Profit_EUR  Margin_%
Transport_Mode                                                              
Bike Share                310355.0           5912287.0  -5601932.0   -1805.0
E-Scooter                 261278.0           3536960.0  -3275681.0   -1254.0
Bus                     12002119.0          11992457.0      9661.0       0.0
Metro                   73334171.0          69029227.0   4304944.0       6.0
Tram                    15666168.0          13355947.0   2310221.0      15.0
Park and Ride           10066524.0           8283138.0   1783386.0      18.0


In [2]:
#Dashboard

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State, ctx, no_update

# Data
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Profit_EUR"] = df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]

# Weighted averages
df["Delay_x_Trips"] = df["Average_Delay_Minutes"] * df["Trips"]
df["OnTime_x_Trips"] = df["On_Time_Percentage"] * df["Trips"]
df["Rating_x_Pax"] = df["Customer_Rating"] * df["Passengers"]
months = sorted(df["Month"].unique())
LAST = len(months) - 1

# Settings
BLUE, ORANGE, GREY, DARK = "#1f77b4", "#ff7f0e", "#6b7280", "#1f2937"
FILTERS = ["City", "Transport_Mode", "Route_Type", "Weather", "Event_Day"]
MEASURES = {                      # label: (numerator column, weight column or None for a total)
    "Passengers": ("Passengers", None),
    "Ticket revenue (EUR)": ("Ticket_Revenue_EUR", None),
    "Profit (EUR)": ("Profit_EUR", None),
    "CO2 saved (kg)": ("CO2_Saved_KG", None),
    "Average delay (min)": ("Delay_x_Trips", "Trips"),
    "Customer rating": ("Rating_x_Pax", "Passengers"),
    "On-time (%)": ("OnTime_x_Trips", "Trips"),
}

def measure_by(d, by, label):
    """Total, or weighted average, of the chosen measure for each group."""
    num, den = MEASURES[label]
    g = d.groupby(by)[[num] + ([den] if den else [])].sum()
    return (g[num] / g[den] if den else g[num]).rename("value").reset_index()

def subset(vals, period, skip=()):
    """Filter by the month range and every filter except those in 'skip'."""
    d = df[(df["Month"] >= months[period[0]]) & (df["Month"] <= months[period[1]])]
    for col, chosen in zip(FILTERS, vals):
        if chosen and col not in skip:
            d = d[d[col].isin(chosen)]
    return d

def empty_fig(msg="No data for this selection"):
    f = go.Figure()
    f.update_layout(template="plotly_white", annotations=[dict(text=msg, showarrow=False)],
                    xaxis_visible=False, yaxis_visible=False, margin=dict(l=10, r=10, t=10, b=10))
    return f

# Layout
def card(title, value, sub=""):
    return html.Div([html.Div(title, style={"fontSize": "12px", "color": GREY}),
                     html.Div(value, style={"fontSize": "24px", "fontWeight": "700", "color": DARK}),
                     html.Div(sub, style={"fontSize": "11px", "color": GREY})],
                    style={"flex": "1", "padding": "10px", "background": "white", "borderRadius": "8px",
                           "boxShadow": "0 1px 3px rgba(0,0,0,.12)", "textAlign": "center"})

def panel(graph_id, question):
    return html.Div([html.Div(question, style={"fontSize": "12px", "color": GREY, "marginBottom": "2px"}),
                     dcc.Graph(id=graph_id, style={"height": "300px"})],
                    style={"background": "white", "borderRadius": "8px", "padding": "10px",
                           "boxShadow": "0 1px 3px rgba(0,0,0,.12)"})

def dropdown(label, column):
    return html.Div([html.Label(label, style={"fontSize": "12px", "fontWeight": "600"}),
                     dcc.Dropdown(id=column, options=sorted(df[column].unique()),
                                  multi=True, placeholder="All")], style={"flex": "1"})

BTN = {"padding": "8px 14px", "border": "none", "borderRadius": "6px", "color": "white",
       "cursor": "pointer", "fontWeight": "600", "marginLeft": "8px"}

app = Dash(__name__)
app.layout = html.Div([
    html.Div([html.H2("EcoMove Sustainable Mobility Dashboard", style={"margin": "0", "color": "white"}),
              html.Div("Decision support for route planning, reliability, sustainability, accessibility and investment",
                       style={"color": "#dbeafe", "fontSize": "13px"})],
             style={"background": DARK, "padding": "14px 20px", "borderRadius": "8px"}),

    # Row 1: dropdown filters
    html.Div([dropdown("City", "City"), dropdown("Transport Mode", "Transport_Mode"),
              dropdown("Route Type", "Route_Type"), dropdown("Weather", "Weather"),
              dropdown("Event Day", "Event_Day")],
             style={"display": "flex", "gap": "12px", "margin": "14px 0 8px"}),

    # Row 2: measure selector, time slider, buttons
    html.Div([
        html.Div([html.Label("Measure for trend and city charts", style={"fontSize": "12px", "fontWeight": "600"}),
                  dcc.Dropdown(id="measure", options=list(MEASURES), value="Passengers", clearable=False)],
                 style={"width": "24%"}),
        html.Div([html.Label("Time period", style={"fontSize": "12px", "fontWeight": "600"}),
                  dcc.RangeSlider(id="period", min=0, max=LAST, step=1, value=[0, LAST],
                                  marks={i: pd.Timestamp(months[i]).strftime("%b %y") for i in range(0, LAST + 1, 3)},
                                  allowCross=False)],
                 style={"flex": "1", "padding": "0 16px"}),
        html.Div([html.Button("Reset all", id="reset", style={**BTN, "background": GREY}),
                  html.Button("Download data (CSV)", id="dl_btn", style={**BTN, "background": BLUE}),
                  dcc.Download(id="dl")], style={"width": "22%", "textAlign": "right", "paddingTop": "14px"}),
    ], style={"display": "flex", "alignItems": "flex-start", "marginBottom": "8px"}),

    html.Div(id="active", style={"fontSize": "13px", "color": DARK, "marginBottom": "8px"}),
    html.Div("Tip: click a city bar, a mode (box plot or profit bar) or a heatmap cell to filter the whole dashboard. "
             "Click it again to clear.", style={"fontSize": "12px", "color": GREY, "marginBottom": "8px"}),

    html.Div(id="kpis", style={"display": "flex", "gap": "10px", "marginBottom": "12px"}),
    html.Div(id="insight", style={"background": "#fff7ed", "borderLeft": f"4px solid {ORANGE}", "padding": "10px 14px",
                                  "borderRadius": "4px", "marginBottom": "12px", "fontSize": "14px", "color": DARK}),

    html.Div([panel("trend", "Route planning: how does the chosen measure change over time?"),
              panel("city_bar", "Investment: how do cities compare on the chosen measure? (click to select)"),
              panel("delay_box", "Reliability: which modes run late, and how consistently? (click to select)"),
              panel("profit_bar", "Investment: which modes pay for themselves? (click to select)"),
              panel("access_heat", "Accessibility: where are complaints highest per passenger? (click a cell)"),
              panel("co2_scatter", "Sustainability: how much CO2 is saved per kWh used?")],
             style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "12px"}),
], style={"fontFamily": "Arial, sans-serif", "padding": "16px", "background": "#f3f4f6"})

# Filters
def toggle(current, value):
    """Clicking the already-selected item clears it."""
    return None if current == [value] else [value]

@app.callback(
    [Output(c, "value") for c in FILTERS] + [Output("period", "value")],
    [Input("city_bar", "clickData"), Input("delay_box", "clickData"), Input("profit_bar", "clickData"),
     Input("access_heat", "clickData"), Input("reset", "n_clicks")],
    [State("City", "value"), State("Transport_Mode", "value"), State("Route_Type", "value")],
    prevent_initial_call=True)
def clicks(city_c, box_c, profit_c, heat_c, reset, city, mode, route):
    out = [no_update] * 6
    t = ctx.triggered_id
    if t == "reset":
        return [None] * 5 + [[0, LAST]]
    if t == "city_bar" and city_c:
        out[0] = toggle(city, city_c["points"][0]["y"])
    elif t == "delay_box" and box_c:
        out[1] = toggle(mode, box_c["points"][0]["x"])
    elif t == "profit_bar" and profit_c:
        out[1] = toggle(mode, profit_c["points"][0]["x"])
    elif t == "access_heat" and heat_c:
        p = heat_c["points"][0]
        out[0], out[2] = toggle(city, p["y"]), toggle(route, p["x"])
    return out

# Fixing the Filters
@app.callback(
    [Output("kpis", "children"), Output("active", "children"), Output("insight", "children"),
     Output("trend", "figure"), Output("city_bar", "figure"), Output("delay_box", "figure"),
     Output("profit_bar", "figure"), Output("access_heat", "figure"), Output("co2_scatter", "figure")],
    [Input(c, "value") for c in FILTERS] + [Input("period", "value"), Input("measure", "value")])
def update(city, mode, route, weather, event, period, measure):
    vals = [city, mode, route, weather, event]
    lay = dict(template="plotly_white", margin=dict(l=40, r=10, t=10, b=40), font=dict(size=11))

    # Showing what is currently selected
    chosen = [f"{n.replace('_', ' ')}: {', '.join(v)}" for n, v in zip(FILTERS, vals) if v]
    span = f"{pd.Timestamp(months[period[0]]):%b %Y} to {pd.Timestamp(months[period[1]]):%b %Y}"
    active = "Showing: " + (" | ".join(chosen) if chosen else "all cities, modes, route types, weather and days") + f" | {span}"

    d = subset(vals, period)
    if d.empty:
        e = empty_fig()
        return [card("No data", "-")], active, "No records match these filters.", e, e, e, e, e, e

    # KPI cards
    pax, profit = d["Passengers"].sum(), d["Profit_EUR"].sum()
    kpis = [card("Passengers", f"{pax/1e6:.2f}M", "total demand"),
            card("Profit", f"€{profit/1e6:.2f}M", "revenue - cost"),
            card("Avg Delay", f"{d['Delay_x_Trips'].sum()/d['Trips'].sum():.1f} min", "per trip (weighted)"),
            card("On-Time", f"{d['OnTime_x_Trips'].sum()/d['Trips'].sum():.1f}%", "trip-weighted"),
            card("Rating", f"{d['Rating_x_Pax'].sum()/d['Passengers'].sum():.2f} / 5", "passenger-weighted"),
            card("CO2 Saved", f"{d['CO2_Saved_KG'].sum()/1e6:.2f} kt", "environment"),
            card("Complaints", f"{d['Accessibility_Complaints'].sum()/pax*1000:.2f}", "per 1,000 pax")]

    wm = measure_by(d, "Transport_Mode", "Average delay (min)")
    worst = wm.loc[wm["value"].idxmax(), "Transport_Mode"]
    top_city = d.groupby("City")["Passengers"].sum().idxmax()
    insight = (f"Key insight: {top_city} has the highest demand in this view, '{worst}' has the longest "
               f"average delay, and the selection is {'profitable' if profit >= 0 else 'loss-making'} (€{profit/1e6:.2f}M).")

    # 1. Trend of the chosen measure
    m = measure_by(d, "Month", measure)
    trend = px.line(m, x="Month", y="value", markers=True, color_discrete_sequence=[BLUE])
    trend.update_layout(**lay, xaxis_title=None, yaxis_title=measure)

    # 2. City comparison
    dc = subset(vals, period, skip=("City",))
    c = measure_by(dc, "City", measure).sort_values("value")
    colours = [ORANGE if (city and x in city) else BLUE for x in c["City"]]
    city_bar = go.Figure(go.Bar(x=c["value"], y=c["City"], orientation="h", marker_color=colours))
    city_bar.update_layout(**lay, xaxis_title=measure, yaxis_title=None)

    # 3. Delay distribution
    dm = subset(vals, period, skip=("Transport_Mode",))
    cmap = {t: (ORANGE if (mode and t in mode) else BLUE) for t in dm["Transport_Mode"].unique()}
    delay_box = px.box(dm, x="Transport_Mode", y="Average_Delay_Minutes", color="Transport_Mode",
                       color_discrete_map=cmap)
    delay_box.update_layout(**lay, xaxis_title=None, yaxis_title="Delay per record (min)", showlegend=False)

    # 4. Profit by mode
    p = dm.groupby("Transport_Mode")["Profit_EUR"].sum().reset_index()
    fade = [1 if (not mode or t in mode) else 0.3 for t in p["Transport_Mode"]]
    profit_bar = go.Figure(go.Bar(x=p["Transport_Mode"], y=p["Profit_EUR"],
                                  marker_color=[BLUE if v >= 0 else ORANGE for v in p["Profit_EUR"]],
                                  marker_opacity=fade))
    profit_bar.update_layout(**lay, xaxis_title=None, yaxis_title="Profit (EUR)  [blue = profit, orange = loss]")

    # 5. Heatmap
    dh = subset(vals, period, skip=("City", "Route_Type"))
    a = dh.groupby(["City", "Route_Type"])[["Accessibility_Complaints", "Passengers"]].sum()
    a = (a["Accessibility_Complaints"] / a["Passengers"] * 1000).unstack().round(2)
    access_heat = px.imshow(a, text_auto=True, color_continuous_scale="Oranges", aspect="auto")
    access_heat.update_layout(**lay, coloraxis_showscale=False, xaxis_title=None, yaxis_title=None)

    # 6. CO2 saved vs energy used
    co2 = px.scatter(d, x="Energy_Consumption_KWh", y="CO2_Saved_KG", color="Transport_Mode",
                     opacity=0.6, color_discrete_sequence=px.colors.qualitative.Safe)
    co2.update_layout(**lay, xaxis_title="Energy (kWh)", yaxis_title="CO2 saved (kg)",
                      legend=dict(title=None, font=dict(size=9)))

    return kpis, active, insight, trend, city_bar, delay_box, profit_bar, access_heat, co2

# Download the filtered data
@app.callback(Output("dl", "data"), Input("dl_btn", "n_clicks"),
              [State(c, "value") for c in FILTERS] + [State("period", "value")], prevent_initial_call=True)
def download(n, city, mode, route, weather, event, period):
    d = subset([city, mode, route, weather, event], period).drop(columns=["Month"])
    return dcc.send_data_frame(d.to_csv, "ecomove_filtered_data.csv", index=False)

if __name__ == "__main__":
    app.run(debug=False)

<IPython.core.display.Javascript object>